# Shortest tour of 14 cities by road

A delivery truck leaves Los Angeles, visits 13 other cities of the United States, and comes back. In which order should it visit them to drive the fewest kilometres? This question is the **travelling salesman problem**.

This notebook answers it in two steps:

1. scgraph computes the road distance between each pair of cities: a **distance matrix**.
2. [PuLP](https://coin-or.github.io/pulp/), a modelling library of [COIN-OR](https://www.coin-or.org/), finds the shortest tour from this matrix.

You need scgraph, and PuLP 4 or newer with a solver. The extra `highs` of PuLP installs the free solver [HiGHS](https://highs.dev). The notebook downloads the network `north_america` once, in its file without the shapes of the arcs (19.0 MB). It then runs in less than one minute.

In [1]:
# Until scgraph 4.0 is on PyPI, install it from the `dev` branch:
# %pip install "scgraph @ git+https://github.com/connor-makowski/scgraph@dev" "pulp[highs]>=4"
import pulp

from scgraph import GeoGraph

## 1. Load the roads of North America

`north_america` is a **multimodal network**: it joins the roads, the railways, the sea lanes and the flights. Each node has a **mode**: `road`, `rail`, `sea`, `port`, `airport` or `rail_terminal`.

`modes=["road"]` loads the road nodes only, and the arcs between them. A truck needs this. With all the modes, a shortest route may take a flight, because a flight is close to a straight line. The roads include the ferries that carry vehicles.

`shapes=False` loads the file without the shapes of the arcs. A shape only serves to draw an arc on a map. The lengths stay the same.

In [2]:
roads = GeoGraph.load_geograph("north_america", modes=["road"], shapes=False)
print(f"{len(roads.nodes):,} road nodes")

603,740 road nodes


## 2. The distance matrix

`distance_matrix` takes a list of places. It returns the length of the shortest route between each pair of places, in kilometres. It computes the routes from one origin together, so it is much faster than one call to `get_shortest_path` for each pair.

In [3]:
CITIES = {
    "Los Angeles": {"latitude": 34.0522, "longitude": -118.2437},
    "New York City": {"latitude": 40.7128, "longitude": -74.0060},
    "Chicago": {"latitude": 41.8781, "longitude": -87.6298},
    "Houston": {"latitude": 29.7604, "longitude": -95.3698},
    "Atlanta": {"latitude": 33.7490, "longitude": -84.3880},
    "Miami": {"latitude": 25.7617, "longitude": -80.1918},
    "Seattle": {"latitude": 47.6062, "longitude": -122.3321},
    "Minneapolis": {"latitude": 44.9778, "longitude": -93.2650},
    "Denver": {"latitude": 39.7392, "longitude": -104.9903},
    "San Francisco": {"latitude": 37.7749, "longitude": -122.4194},
    "Boston": {"latitude": 42.3601, "longitude": -71.0589},
    "Washington DC": {"latitude": 38.9072, "longitude": -77.0369},
    "Charleston": {"latitude": 32.7765, "longitude": -79.9311},
    "New Orleans": {"latitude": 29.9511, "longitude": -90.0715},
}
names = list(CITIES)

distance_matrix = roads.distance_matrix(list(CITIES.values()), output_units="km")
print(f"Los Angeles to New York City: {distance_matrix[0][1]:,.1f} km")
print(f"Los Angeles to San Francisco: {distance_matrix[0][9]:,.1f} km")

Los Angeles to New York City: 4,392.7 km
Los Angeles to San Francisco: 613.5 km


## 3. The shortest tour

The function below writes the problem as a **mixed integer linear program**. For each pair of cities `i` and `j`, a variable `x[i][j]` is 1 if the tour goes from `i` to `j`, and 0 if not. The constraints are:

- The tour leaves each city once, and enters each city once.
- The tour is one loop, not several small loops. The variables `order[i]` give the rank of each city in the tour, and forbid small loops. This is the formulation of Miller, Tucker and Zemlin.

The solver HiGHS then finds the variables that give the shortest tour. To use the solver CBC of COIN-OR instead, install `pulp[cbc]` (a download of about 170 MB) and replace `pulp.HiGHS` by `pulp.COIN_CMD`.

In [4]:
def solve_tsp(distance_matrix):
    """The shortest tour that visits each city once and comes back to city 0.

    `distance_matrix[i][j]` is the distance from city i to city j. Returns the
    list of the cities of the tour, from city 0 back to city 0, and its length.
    """
    n = len(distance_matrix)
    cities = range(n)
    problem = pulp.LpProblem("TSP", pulp.LpMinimize)

    # x[i][j] is 1 if the tour goes from city i to city j
    x = problem.add_variable_dicts("x", (cities, cities), cat="Binary")
    # order[i] is the rank of city i in the tour; it forbids small loops
    order = problem.add_variable_dicts("order", cities, lowBound=0, upBound=n - 1)

    problem += pulp.lpSum(
        distance_matrix[i][j] * x[i][j] for i in cities for j in cities if i != j
    )
    for i in cities:
        problem += pulp.lpSum(x[i][j] for j in cities if i != j) == 1  # leave once
        problem += pulp.lpSum(x[j][i] for j in cities if i != j) == 1  # enter once
    for i in cities:
        for j in cities:
            if i != j and i != 0 and j != 0:
                problem += order[i] - order[j] + (n - 1) * x[i][j] <= n - 2

    problem.solve(pulp.HiGHS(msg=False))

    tour = [0]
    while len(tour) < n:
        tour.append(next(j for j in cities if j != tour[-1] and pulp.value(x[tour[-1]][j]) > 0.5))
    tour.append(0)
    return tour, pulp.value(problem.objective)


tour, length_km = solve_tsp(distance_matrix)
print(f"Shortest tour: {length_km:,.0f} km")
for city in tour:
    print("    ", names[city])

Running HiGHS 1.16.0 (git hash: f3f113d): Copyright (c) 2026 under MIT licence terms
Includes third-party software components, see THIRD_PARTY_NOTICES.md for full details


Shortest tour: 14,785 km
     Los Angeles
     Denver
     Houston
     New Orleans
     Atlanta
     Miami
     Charleston
     Washington DC
     New York City
     Boston
     Chicago
     Minneapolis
     Seattle
     San Francisco
     Los Angeles


From Los Angeles, the tour goes east through Denver to Houston. It follows the Gulf of Mexico to Miami, and goes up the east coast to Boston. It comes back west through Chicago and Minneapolis to Seattle, and goes down the west coast.

scgraph routes by distance. [Issue #287](https://github.com/connor-makowski/scgraph/issues/287) tracks routing by time, cost and CO2.